# Moduł 9: Spark Structured Streaming

W Module 8 poznaliśmy Apache Kafkę -- system, który **doprowadza** dane strumieniowe od
producentów do konsumentów. Ale kto właściwie te dane **przetwarza**? W wielu nowoczesnych
architekturach odpowiedzią jest **Apache Spark Structured Streaming** -- silnik
przetwarzania strumieniowego zbudowany na dokładnie tym samym DataFrame API, które
poznaliśmy w kursie Databricks/PySpark.

To dobra wiadomość: **prawie wszystko, czego nauczyłeś się o DataFrame'ach, transformacjach
i agregacjach, przenosi się bezpośrednio na streaming.** Structured Streaming to nie osobny
język ani osobne API -- to ten sam kod, uruchomiony w trybie ciągłym.

W przeciwieństwie do Modułu 8, ten moduł jest w pełni **wykonywalny lokalnie** -- Spark
potrafi symulować źródła strumieniowe (`rate`, pliki w katalogu) bez żadnej zewnętrznej
infrastruktury, więc każdą demonstrację uruchomimy naprawdę i zobaczymy prawdziwe wyniki.

**Spis treści:**
1. Od batchu do streamingu: model mikro-wsadowy
2. Źródła danych strumieniowych -- przegląd
3. Pierwszy strumień: źródło `rate` i tryby wyjścia
4. Czas zdarzenia, okna czasowe i znaki wodne (watermark)
5. Przetwarzanie przyrostowe plików wejściowych
6. `foreachBatch` -- łączenie streamingu z logiką wsadową
7. Źródło Kafka w praktyce (składnia)
8. Podsumowanie i ćwiczenia

## 1. Od batchu do streamingu: model mikro-wsadowy

W przetwarzaniu **wsadowym (batch)** masz kompletny, ograniczony zbiór danych -- czytasz go,
przekształcasz, zapisujesz wynik i kończysz. W przetwarzaniu **strumieniowym** dane napływają
bez końca, a Ty chcesz na bieżąco aktualizować wynik w miarę ich przybywania.

Structured Streaming rozwiązuje to za pomocą prostej, ale potężnej idei: **traktuje strumień
jako nieograniczoną (unbounded) tabelę**, do której w tle ciągle dopisywane są nowe wiersze.
Twoje zapytanie (`select`, `groupBy`, `join`...) jest napisane tak, jakby operowało na zwykłej
tabeli -- a silnik Sparka sam odpowiada za uruchamianie go raz po raz na nowych porcjach
danych i łączenie wyniku z tym, co już policzył wcześniej.

To podejście nazywa się **modelem mikro-wsadowym (micro-batch)**: zamiast przetwarzać każdy
rekord z osobna (jak np. Flink czy Kafka Streams w trybie natywnym), Spark grupuje nowo
przybyłe dane w małe wsady (np. co kilka sekund) i przetwarza każdy wsad, używając tego
samego silnika co przy zwykłych DataFrame'ach.

> 💡 **Dlaczego to ważne**
>
> Dzięki modelowi mikro-wsadowemu ten sam kod (`filter`, `groupBy`, `join`, UDF-y) działa identycznie w trybie batch i streaming. Nie musisz uczyć się nowego API -- musisz tylko zrozumieć kilka nowych pojęć: źródła strumieniowe, tryby wyjścia, checkpointing i czas zdarzenia.

> ⚡ **Continuous Processing**
>
> Spark ma też eksperymentalny tryb `Continuous Processing`, który przetwarza rekord po rekordzie z bardzo niskim opóźnieniem (milisekundy zamiast sekund), ale wspiera tylko ograniczony zestaw operacji i rzadko jest używany w praktyce. W tym module -- i w zdecydowanej większości rzeczywistych wdrożeń -- używamy domyślnego trybu mikro-wsadowego.

Uruchamiamy `SparkSession` -- dokładnie tak samo jak w kursie Databricks/PySpark. Dodatkowo ograniczamy liczbę partycji przy tasowaniu (`shuffle.partitions`) do małej wartości -- domyślne 200 partycji ma sens na klastrze produkcyjnym, ale przy naszych malutkich strumieniach testowych tylko spowalnia każdy mikro-wsad.

In [ ]:
from pyspark.sql import SparkSession

# Dodatkowe opcje JVM (--add-opens) sa potrzebne, gdy uruchamiasz Sparka lokalnie na
# Javie 17+ i chcesz uzywac pandas UDF / Apache Arrow (Modul 8) -- nowsze JDK domyslnie
# blokuja refleksyjny dostep do pamieci poza stosem, ktorego potrzebuje Arrow. W
# Databricks nie trzeba tego ustawiac recznie -- platforma robi to za Ciebie.
_opcje_jvm = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("kurs_spark")
    .config("spark.driver.extraJavaOptions", _opcje_jvm)
    .config("spark.executor.extraJavaOptions", _opcje_jvm)
    .getOrCreate()
)
spark


In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "2")
print("Gotowe -- SparkSession skonfigurowana do pracy ze strumieniami.")

## 2. Źródła danych strumieniowych -- przegląd

Spark potrafi czytać strumienie z kilku typów źródeł. Poniższa tabela porównuje te, które
napotkasz najczęściej:

| Źródło | Do czego służy | Dostępne w tym środowisku? |
|---|---|---|
| `rate` | Generuje syntetyczne dane (licznik + znacznik czasu) ze stałą prędkością -- idealne do testów i nauki | ✅ tak |
| `file` | Czyta nowe pliki pojawiające się w katalogu (np. landing zone z danymi z zewnętrznego systemu) | ✅ tak |
| `socket` | Czyta linie tekstu z gniazda TCP -- używane głównie do testów i demonstracji | ⚠️ wymaga własnego serwera TCP (pomijamy w tym module) |
| `kafka` | Czyta z tematów Apache Kafka -- **standard produkcyjny** dla prawdziwego streamingu | ❌ brak brokera Kafka (patrz Moduł 8) |

W tym module skupimy się na `rate` (do nauki mechaniki okien i trybów wyjścia) oraz `file`
(bo bardzo dobrze oddaje częsty scenariusz produkcyjny: dane trafiają jako pliki do
"strefy lądowania", a Spark przetwarza je przyrostowo). Na końcu modułu pokażemy dokładną
składnię źródła `kafka` -- to właśnie nią połączyłbyś ten kod z tematem `zamowienia`
z Modułu 8 na prawdziwym klastrze.

## 3. Pierwszy strumień: źródło `rate` i tryby wyjścia

Każdy strumień w Structured Streaming składa się z trzech elementów:

1. **Źródło** (`readStream...`) -- skąd napływają dane.
2. **Transformacja** -- zwykłe operacje DataFrame (`select`, `filter`, `groupBy`...).
3. **Ujście (sink)** i **tryb wyjścia** (`writeStream...outputMode(...)`) -- gdzie i jak
   zapisywany jest wynik.

Tryby wyjścia (`outputMode`) określają, co dokładnie trafia do ujścia przy każdym
mikro-wsadzie:

| Tryb | Co zapisuje | Kiedy używać |
|---|---|---|
| `append` | Tylko nowe wiersze, które nie zmienią się już w przyszłości | Dane bez agregacji, albo okna czasowe po zamknięciu (patrz sekcja 4) |
| `update` | Wiersze, które zmieniły się od ostatniego wsadu | Bieżące, ciągle aktualizowane agregacje (np. licznik na żywo) |
| `complete` | Cała zaktualizowana tabela wyniku za każdym razem | Małe agregacje, gdzie chcesz zawsze widzieć pełny obraz |

Zacznijmy od najprostszego strumienia: źródła `rate`, które generuje wiersze
(`timestamp`, `value`) ze stałą prędkością, zapisane do **ujścia pamięciowego** (`memory`)
-- wygodnego do nauki, bo możemy odpytywać wynik zwykłym SQL-em w tym samym notebooku.

In [ ]:
import time

strumien = spark.readStream.format("rate").option("rowsPerSecond", 5).load()
strumien.printSchema()

zapytanie = (
    strumien.writeStream
    .format("memory")
    .queryName("rate_demo")
    .outputMode("append")
    .start()
)

time.sleep(4)
zapytanie.stop()

spark.sql("SELECT COUNT(*) AS liczba_wierszy FROM rate_demo").show()
spark.sql("SELECT * FROM rate_demo ORDER BY value LIMIT 5").show()

> ⚠️ **Zawsze zatrzymuj zapytanie**
>
> Zapytanie strumieniowe (`StreamingQuery`) domyślnie działa **w nieskończoność** w tle -- w prawdziwej aplikacji to pożądane, ale w notebooku musisz jawnie wywołać `.stop()`, tak jak zrobiliśmy powyżej. Zapomniane, niezatrzymane zapytanie będzie zużywać zasoby klastra w nieskończoność.

> 🏭 **Checkpointing**
>
> Powyższy przykład nie ustawia `checkpointLocation` -- w ujściu `memory` nie jest to wymagane, bo i tak nic nie przetrwa restartu notebooka. Na produkcji **każde** zapytanie strumieniowe powinno mieć ustawiony checkpoint (`.option("checkpointLocation", ...)`) -- to właśnie tam Spark zapisuje, które dane już przetworzył, żeby po awarii wznowić pracę dokładnie tam, gdzie skończył, bez duplikatów i bez utraty danych. Zobaczymy to w akcji w sekcji 5.

## 4. Czas zdarzenia, okna czasowe i znaki wodne (watermark)

W analizach strumieniowych prawie zawsze chcesz agregować dane **w oknach czasowych** --
np. "ile zamówień na minutę", "suma sprzedaży co 5 minut". Structured Streaming wspiera to
natywnie przez `groupBy(window(...))`.

Pojawia się jednak pytanie: kiedy okno jest "zamknięte" i można bezpiecznie wypisać jego
ostateczny wynik? Dane mogą przecież docierać z opóźnieniem (np. urządzenie IoT chwilowo
straciło połączenie). Odpowiedzią jest **znak wodny (watermark)** -- deklaracja "ile
maksymalnie spóźnienia akceptuję, zanim uznam okno za ostatecznie zamknięte i pozwolę
Sparkowi zwolnić pamięć trzymaną na jego temat".

In [ ]:
from pyspark.sql import functions as F

strumien2 = spark.readStream.format("rate").option("rowsPerSecond", 10).load()

zliczenia_w_oknach = (
    strumien2
    .withWatermark("timestamp", "5 seconds")
    .groupBy(F.window("timestamp", "5 seconds"))
    .count()
)

zapytanie2 = (
    zliczenia_w_oknach.writeStream
    .format("memory")
    .queryName("okna_demo")
    .outputMode("update")
    .start()
)

time.sleep(8)
zapytanie2.stop()

spark.sql("SELECT * FROM okna_demo ORDER BY window").show(truncate=False)

> ⚡ **Dlaczego `update`, a nie `append`?**
>
> W trybie `update` widzimy, jak licznik danego okna rośnie z każdym mikro-wsadem -- dokładnie to, czego potrzebujesz do dashboardu "na żywo". Tryb `append` dla zapytań z oknami wypisze wiersz **dopiero gdy watermark przekroczy koniec okna** (czyli gdy Spark jest już pewien, że nic więcej do tego okna nie dotrze) -- to z kolei świetnie nadaje się do zapisu do trwałego, dołączanego magazynu danych (np. tabeli faktów), gdzie nie chcesz nadpisywać już zapisanych wierszy.

> ⚠️ **Watermark a spóźnione dane**
>
> Jeśli rekord dotrze z czasem zdarzenia **starszym** niż aktualny watermark, Spark go **odrzuci** -- to świadomy kompromis: bez takiej granicy silnik musiałby trzymać w pamięci stan dla wszystkich okien od początku istnienia strumienia. Dobór długości watermarka to decyzja biznesowa: "ile spóźnienia jestem w stanie zaakceptować, zanim zamknę okno na zawsze?"

## 5. Przetwarzanie przyrostowe plików wejściowych

Bardzo częsty scenariusz produkcyjny: zewnętrzny system (albo Kafka Connect, patrz Moduł 8)
zrzuca pliki (JSON, CSV, Parquet) do katalogu na obiektowym magazynie (S3, ADLS...), a Spark
ma je przetwarzać **przyrostowo** -- czyli za każdym razem tylko te pliki, których jeszcze
nie widział.

Do tego służy **trigger `availableNow`**: zamiast działać w nieskończoność, zapytanie
przetwarza wszystkie aktualnie dostępne, jeszcze nie przetworzone dane, a potem **samo się
kończy**. To bardzo wygodne w notebooku (nie trzeba `sleep` + `stop`) i często używane też
na produkcji jako tańsza alternatywa dla stale działającego klastra streamingowego --
uruchamiasz taki job np. co godzinę przez Airflow (Moduł 2-3!) zamiast trzymać klaster 24/7.

Kluczowy element: **checkpoint**. To właśnie w checkpointcie Spark zapisuje listę już
przetworzonych plików -- dzięki temu drugie uruchomienie tego samego zapytania nie
przetworzy ponownie plików z pierwszego wsadu.

In [ ]:
import os
import json
import shutil

katalog_zrodlowy = os.path.abspath("strumien_zamowien")
katalog_checkpoint = os.path.abspath("checkpoint_m9")
shutil.rmtree(katalog_zrodlowy, ignore_errors=True)
shutil.rmtree(katalog_checkpoint, ignore_errors=True)
os.makedirs(katalog_zrodlowy, exist_ok=True)

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schemat_zamowien = StructType([
    StructField("zamowienie_id", IntegerType()),
    StructField("klient_id", IntegerType()),
    StructField("miasto", StringType()),
    StructField("wartosc", DoubleType()),
])


def zapisz_plik_wejsciowy(nazwa, wiersze):
    """Symuluje zewnetrzny system dopisujacy nowy plik do strefy ladowania."""
    with open(f"{katalog_zrodlowy}/{nazwa}", "w") as f:
        for w in wiersze:
            f.write(json.dumps(w) + "\n")


# --- Partia 1: pierwsze dwa zamowienia docieraja do strefy ladowania ---
zapisz_plik_wejsciowy("czesc_1.json", [
    {"zamowienie_id": 1, "klient_id": 10, "miasto": "Warszawa", "wartosc": 100.0},
    {"zamowienie_id": 2, "klient_id": 11, "miasto": "Krakow", "wartosc": 200.0},
])

strumien_plikow = spark.readStream.schema(schemat_zamowien).json(katalog_zrodlowy)
suma_wg_miasta = strumien_plikow.groupBy("miasto").sum("wartosc")

zapytanie3 = (
    suma_wg_miasta.writeStream
    .format("memory")
    .queryName("suma_zamowien")
    .outputMode("complete")
    .option("checkpointLocation", katalog_checkpoint)
    .trigger(availableNow=True)
    .start()
)
zapytanie3.awaitTermination()

print("--- Po partii 1 ---")
spark.sql("SELECT * FROM suma_zamowien ORDER BY miasto").show()

In [ ]:
# --- Partia 2: kolejne zamowienia docieraja z opoznieniem ---
zapisz_plik_wejsciowy("czesc_2.json", [
    {"zamowienie_id": 3, "klient_id": 10, "miasto": "Warszawa", "wartosc": 50.0},
    {"zamowienie_id": 4, "klient_id": 12, "miasto": "Gdansk", "wartosc": 300.0},
])

# Uruchamiamy TO SAMO zapytanie ponownie -- dzieki checkpointowi Spark wie, ze "czesc_1.json"
# zostal juz przetworzony, i doda tylko "czesc_2.json" do istniejacego stanu agregacji.
zapytanie4 = (
    suma_wg_miasta.writeStream
    .format("memory")
    .queryName("suma_zamowien")
    .outputMode("complete")
    .option("checkpointLocation", katalog_checkpoint)
    .trigger(availableNow=True)
    .start()
)
zapytanie4.awaitTermination()

print("--- Po partii 2 (Warszawa: 100+50, plus nowy Gdansk) ---")
spark.sql("SELECT * FROM suma_zamowien ORDER BY miasto").show()

> 💡 **Co się właśnie stało**
>
> Warszawa ma sumę **150** (100 z partii 1 + 50 z partii 2), mimo że drugie zapytanie widziało tylko plik `czesc_2.json`. To zasługa checkpointu: Spark pamięta zarówno *które pliki przetworzył*, jak i *stan dotychczasowej agregacji*, więc może kontynuować liczenie od miejsca, w którym skończył -- dokładnie tak, jak zrobiłby to po awarii i restarcie.

## 6. `foreachBatch` -- łączenie streamingu z logiką wsadową

Czasem wbudowane ujścia (`memory`, `console`, `parquet`, `kafka`...) nie wystarczają --
chcesz np. zapisać każdy mikro-wsad do istniejącej tabeli w hurtowni (Moduł 6), wykonać
upsert, albo uruchomić dowolną logikę Pythona/SQL. Do tego służy `foreachBatch`: Twoja
funkcja dostaje **zwykły DataFrame** (statyczny, nie strumieniowy!) reprezentujący bieżący
mikro-wsad -- i możesz go przetworzyć dokładnie tak, jak każdy inny DataFrame w kursie
Databricks/PySpark (`.write.mode(...)`, `.toPandas()`, joiny ze statycznymi tabelami itd.).

In [ ]:
historia_wsadow = []


def przetworz_wsad(batch_df, batch_id):
    liczba = batch_df.count()
    historia_wsadow.append((batch_id, liczba))
    print(f"Mikro-wsad {batch_id}: {liczba} wierszy")
    # W prawdziwej aplikacji tutaj np.:
    #   batch_df.write.mode("append").saveAsTable("hurtownia.fact_zdarzenia")
    # albo wywolanie dbt / zapis do DuckDB / upsert do Delty.


strumien3 = spark.readStream.format("rate").option("rowsPerSecond", 5).load()
zapytanie5 = strumien3.writeStream.foreachBatch(przetworz_wsad).start()

time.sleep(3)
zapytanie5.stop()

print("Historia mikro-wsadow:", historia_wsadow)

> 🏭 **foreachBatch jako most do dbt i Airflow**
>
> `foreachBatch` to często używany wzorzec do **zasilania hurtowni danych w niemal czasie rzeczywistym**: mikro-wsad zapisujesz do tabeli "surowej" (np. `raw.zdarzenia`), a osobny, zaplanowany w Airflow (Moduł 2-3) `dbt run` co jakiś czas przekształca nowo dopisane wiersze w model gwiazdy (Moduł 6). Zobaczymy dokładnie taki połączony potok w projekcie końcowym (Moduł 10).

## 7. Źródło Kafka w praktyce (składnia)

Tak jak w Module 8, w tym środowisku nie mamy dostępu do działającego brokera Kafka -- ale
warto znać dokładną składnię, bo to właśnie nią połączyłbyś dzisiejszy kod z tematem
`zamowienia` z Modułu 8 na prawdziwym klastrze. Poniższy kod jest gotowy do uruchomienia
(wymaga tylko dodania pakietu `spark-sql-kafka-0-10` przy starcie Sparka), ale w tym
notebooku pozostaje przykładem do przeczytania.

```python
# Wymaga uruchomienia Sparka z pakietem, np.:
#   pyspark --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0

strumien_kafka = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "zamowienia")          # nazwa topicu z Modulu 8
    .option("startingOffsets", "earliest")      # albo "latest"
    .load()
)

# Kafka dostarcza dane jako pary klucz/wartosc w formacie binarnym (bytes) --
# trzeba je jawnie zdekodowac i sparsowac (tu: JSON) do wlasciwego schematu.
from pyspark.sql import functions as F

zamowienia_z_kafki = (
    strumien_kafka
    .selectExpr("CAST(key AS STRING) AS klient_id", "CAST(value AS STRING) AS json_zamowienia")
    .select("klient_id", F.from_json("json_zamowienia", schemat_zamowien).alias("dane"))
    .select("klient_id", "dane.*")
)

zapytanie_kafka = (
    zamowienia_z_kafki.writeStream
    .format("console")
    .outputMode("append")
    .option("checkpointLocation", "/checkpointy/zamowienia_z_kafki")
    .start()
)
```

> ⚡ **To ten sam kod co dla `rate` i `file`**
>
> Zwróć uwagę: cała logika transformacji (`selectExpr`, `from_json`, dowolny `groupBy` czy `join`, który dopiszesz dalej) jest **identyczna** niezależnie od źródła. Zmienia się tylko `.format("kafka")` zamiast `.format("rate")` czy `.json(katalog)`. To właśnie największa siła Structured Streaming -- ucząc się mechaniki na `rate` i plikach, uczysz się jednocześnie tego, co potrzebne do pracy z Kafką na produkcji.

## 8. Podsumowanie

- **Model mikro-wsadowy**: strumień to nieograniczona tabela; Spark uruchamia Twoje zapytanie
  cyklicznie na nowych porcjach danych, używając tego samego DataFrame API co w batchu.
- **Źródła**: `rate` i `socket` do nauki/testów, `file` do przetwarzania stref lądowania,
  `kafka` to standard produkcyjny (Moduł 8).
- **Tryby wyjścia**: `append` (tylko nowe, niezmienne wiersze), `update` (zmienione od
  ostatniego wsadu), `complete` (cała tabela wyniku za każdym razem).
- **Czas zdarzenia + okna + watermark**: `groupBy(window(...))` agreguje w oknach czasowych;
  `withWatermark` mówi Sparkowi, ile spóźnienia tolerować, zanim okno zostanie zamknięte na
  zawsze i zwolniona zostanie pamięć.
- **Trigger `availableNow` + checkpoint**: pozwala przetwarzać dane przyrostowo i kończyć
  zapytanie automatycznie -- świetne do uruchamiania jako zaplanowany job (Airflow).
- **`foreachBatch`**: most między streamingiem a dowolną logiką wsadową (zapis do tabel,
  wywołanie dbt, upsert).

### 📝 Ćwiczenie 1: Okno w trybie `append`

Zmodyfikuj przykład z sekcji 4 (zliczenia w oknach 5-sekundowych) tak, aby użyć trybu `outputMode("append")` zamiast `"update"`. Uruchom, poczekaj dłużej (np. `time.sleep(15)`) i porównaj wynik z wersją `update`. Ile wierszy pojawia się teraz w tabeli wynikowej i kiedy?

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.sql import functions as F

strumien_ex1 = spark.readStream.format("rate").option("rowsPerSecond", 10).load()

zliczenia_append = (
    strumien_ex1
    .withWatermark("timestamp", "5 seconds")
    .groupBy(F.window("timestamp", "5 seconds"))
    .count()
)

zapytanie_ex1 = (
    zliczenia_append.writeStream
    .format("memory")
    .queryName("okna_append_demo")
    .outputMode("append")
    .start()
)

time.sleep(15)
zapytanie_ex1.stop()

spark.sql("SELECT * FROM okna_append_demo ORDER BY window").show(truncate=False)
```

W trybie `append` okno pojawia się w wyniku dopiero, gdy watermark przekroczy jego koniec (czyli zwykle jedno okno później niż w `update`) -- i pojawia się już z ostateczną, niezmienną wartością, dokładnie raz.

</details>

### 📝 Ćwiczenie 2: Trzecia partia plików

Dopisz do przykładu z sekcji 5 trzecią partię plików (`czesc_3.json`) z nowymi zamówieniami dla miast, które już się pojawiły, oraz jednym zupełnie nowym miastem. Uruchom zapytanie z tym samym checkpointem po raz trzeci i zweryfikuj, że suma dla istniejących miast poprawnie się zwiększyła.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
zapisz_plik_wejsciowy("czesc_3.json", [
    {"zamowienie_id": 5, "klient_id": 13, "miasto": "Warszawa", "wartosc": 25.0},
    {"zamowienie_id": 6, "klient_id": 14, "miasto": "Poznan", "wartosc": 150.0},
])

zapytanie_ex2 = (
    suma_wg_miasta.writeStream
    .format("memory")
    .queryName("suma_zamowien")
    .outputMode("complete")
    .option("checkpointLocation", katalog_checkpoint)
    .trigger(availableNow=True)
    .start()
)
zapytanie_ex2.awaitTermination()

spark.sql("SELECT * FROM suma_zamowien ORDER BY miasto").show()
```

Warszawa powinna teraz mieć sumę 175.0 (100 + 50 + 25), a Poznań pojawi się jako nowy wiersz z wartością 150.0 -- checkpoint pamięta cały dotychczasowy stan agregacji.

</details>

### 📝 Ćwiczenie 3: Wybór źródła

Dla każdego z poniższych scenariuszy wskaż, czy użyłbyś źródła `rate`, `file` czy `kafka`, i krótko uzasadnij: (a) testowanie logiki agregującej na CI przed wdrożeniem, (b) codzienne pliki eksportu z systemu ERP trafiające na S3, (c) zdarzenia kliknięć użytkowników na stronie, docierające w czasie rzeczywistym z wielu serwerów aplikacji.

<details>
<summary>Pokaż rozwiązanie</summary>

```text
(a) rate -- generuje deterministyczny, syntetyczny strumien bez zadnej infrastruktury,
    idealny do szybkich testow automatycznych.
(b) file (z triggerem availableNow) -- pliki juz laduja w znanym katalogu, nie trzeba
    dodatkowego brokera; mozna uruchamiac jako zaplanowany job (Airflow).
(c) kafka -- wysoka czestotliwosc zdarzen z wielu zrodel wymaga bufora odpornego na awarie
    i wielu konsumentow (Modul 8) -- rate/file by sie tu nie sprawdzily.
```

</details>

> 🔥 **Wyzwanie**
>
> Zaprojektuj (opisowo, bez konieczności uruchamiania) potok `foreachBatch`, który dla każdego mikro-wsadu ze strumienia `zamowienia` (Kafka, Moduł 8): (1) zapisuje surowe zdarzenia do tabeli `raw.zamowienia_streaming` w DuckDB, (2) po zapisie wywołuje `dbt run --select stg_zamowienia_streaming+` jako podproces (tak jak robiliśmy to w Module 4-5), żeby natychmiast przebudować zależne modele. Jakie ryzyko niesie wywoływanie `dbt run` przy **każdym** mikro-wsadzie (np. co kilka sekund) i jak mógłbyś je ograniczyć?

<details>
<summary>Pokaż rozwiązanie</summary>

```text
Szkic (opisowy, nie do wykonania):

def przetworz_wsad_zamowien(batch_df, batch_id):
    (batch_df.write.mode("append")
        .format("jdbc")  # lub polaczenie duckdb
        .option(...)
        .saveAsTable("raw.zamowienia_streaming"))
    subprocess.run(["dbt", "run", "--select", "stg_zamowienia_streaming+", ...])

Ryzyko: dbt run ma zauwazalny narzut startowy (parsowanie projektu, planowanie DAG-u) --
wywolywanie go co kilka sekund przy kazdym malutkim mikro-wsadzie zdominowaloby czas
przetwarzania i mogloby przeciazyc baze docelowa rownoleglymi transformacjami.

Ograniczenie: zwiekszyc rozmiar mikro-wsadu (dluzszy trigger processingTime, np. co 5-10
minut) ALBO oddzielic zapis surowych danych (czesty, tani) od uruchamiania dbt (rzadsze,
zaplanowane osobno np. co 15 minut przez Airflow -- dokladnie wzorzec z sekcji 6).
```

</details>

## Co dalej?

W ostatnim module tego kursu połączymy wszystko, czego się nauczyliśmy: **Moduł 10 --
Projekt końcowy** zbuduje kompletny potok danych end-to-end -- DAG w Airflow orkiestrujący
transformacje dbt do modelu gwiazdy (z wymiarem SCD2) na naszym zestawie danych sklepu
internetowego, wzbogacony o komponent Structured Streaming zasilający tabelę faktów w
czasie zbliżonym do rzeczywistego.